In [56]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings_cox"


In [57]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 21 firm-variable mappings across 18 firms.


firm_id,variable,notes,final_choice,num_candidates
HARBb.CO,XSGA_COMPONENTS,"Candidate list missed the key SG&A row 'Sales and distribution costs' and also lacked 'Other operating costs' as an overhead bucket. Chose component overhead rows above EBIT and excluded totals, COGS, depreciation/amortization lines, and balancing rows. Manual review flagged because two chosen rows were outside the candidate list. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Sales and distribution costs, [Income Statement] Other operating costs",Income Statement::Sales and distribution costs; Income Statement::Administrative costs; Income Statement::Other operating costs,8
HHDC.CO,XSGA_COMPONENTS,"Candidate list appears incomplete for SG&A because a clear overhead row 'Sales and distribution costs' exists in the preview but was not included among candidates. Chose it from the full preview per instructions. Also selected 'Administrative costs', but that label appears twice in the sheet, creating ambiguity about which instance is intended; exact-label constraints prevent disambiguation. Excluded subtotal-like 'Sales costs' because it equals the distribution row plus balancing values. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Sales and distribution costs",Income Statement::Sales and distribution costs; Income Statement::Administrative costs; Income Statement::Share-based payment; Income Statement::STAFF EXPENSES,3
HLUNa.CO,XSGA_COMPONENTS,"Candidate list missed the clear overhead subtotal row 'Sales and distribution costs', which is present in the Income Statement and is preferable to its staff-cost subcomponent. Added it from the full preview, so manual review is required. Included 'Research and development costs' in XSGA_COMPONENTS because it is separately reported above EBIT and should also be mapped to XRD so that XSGA_COMPONENTS - XRD excludes R&D. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Sales and distribution costs","Income Statement::Sales and distribution costs; Income Statement::Administrative expenses; Income Statement::Research and development costs; Income Statement::Other operating expenses, net",10
HUSCO.CO,XSGA_COMPONENTS,"No explicit SG&A row exists. Candidate list misses valid personnel component rows present in column A ('Defined contribution pension plans', 'Other social security costs'). Per rules, selected granular overhead components instead of subtotal 'Staff cost'. Manual review flagged because final selection uses rows outside the provided candidate list. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Defined contribution pension plans, [Income Statement] Other social security costs",Income Statement::Wages and salaries; Income Statement::Defined contribution pension plans; Income Statement::Other social security costs; Income Statement::Other staff Costs; Income Statement::Share-based remuneration; Income Statement::Other external expenses,9
ISS.CO,COGS,"Candidate list clearly misses a valid income-statement cost line. The statements do not report an explicit 'Cost of sales'/'COGS' line. For this service-heavy business, 'Employee costs' appears to be the primary direct operating/service-delivery cost, but classification versus overhead is judgmental. Manual review recommended. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Employee costs",Income Statement::Employee costs,1
KBHL.CO,COGS,Candidate list clearly misses any income-statement cost-of-sales line. Chose 'Operating and maintenance costs' from the full preview as the most plausible direct operating/service cost for an airport operator. Manual review needed because classification between direct operating costs and overhead is judgmental here. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Operating and maintenance costs,Income Statement::Operating and maintenance costs,1
KBHL.C

In [58]:
df_review = pd.DataFrame(rows)

df_xsga = (
    df_review[df_review["variable"] == "XSGA_COMPONENTS"]
    .sort_values(["firm_id"])
    .reset_index(drop=True)
)

print(f"XSGA_COMPONENTS manual reviews: {len(df_xsga)} rows across {df_xsga['firm_id'].nunique() if len(df_xsga) else 0} firms.")
display(df_xsga)

XSGA_COMPONENTS manual reviews: 14 rows across 14 firms.


,firm_id,variable,notes,final_choice,num_candidates
0,HARBb.CO,XSGA_COMPONENTS,Candidate list missed the key SG&A row 'Sales ...,Income Statement::Sales and distribution costs...,8
1,HHDC.CO,XSGA_COMPONENTS,Candidate list appears incomplete for SG&A bec...,Income Statement::Sales and distribution costs...,3
2,HLUNa.CO,XSGA_COMPONENTS,Candidate list missed the clear overhead subto...,Income Statement::Sales and distribution costs...,10
3,HUSCO.CO,XSGA_COMPONENTS,No explicit SG&A row exists. Candidate list mi...,Income Statement::Wages and salaries; Income S...,9
4,KLEEb.CO,XSGA_COMPONENTS,Used labels outside the candidate list because...,Income Statement::Distribution Costs; Income S...,4
5,NOVOb.CO,XSGA_COMPONENTS,Candidate list misses the best explicit SG&A r...,Income Statement::Sales and distribution costs...,14
6,NSISb.CO,XSGA_COMPONENTS,Candidate list misses the important overhead r...,Income Statement::Sales and distribution expen...,3
7,PAALb.CO,XSGA_COMPONENTS,Explicit SG&A row exists but appears unpopulat...,"Income Statement::Selling, General & Admin; In...",6
8,RBREW.CO,XSGA_COMPONENTS,Candidate list missed the clearly relevant row...,Income Statement::Other Sales and distribution...,9
9,ROCKb.CO,XSGA_COMPONENTS,No explicit SG&A row is present. Selected the ...,Income Statement::Employee benefits expenses; ...,5
